In [5]:
import json
import torch
import numpy as np



In [ ]:
with open('OMNI_1hr.json', 'r') as file:
    data_1hr = json.load(file)

print(data_1hr)

In [ ]:
#fonctions pour préparer les données 

def gat_rand_mask(observed_mask):
    """Masquage aléatoire pour l'entraînement. Prend le masque des vraies observations et masque une proportion aléatoire pour créer 
    le masque conditionnel (m^co) qui sert pour l'apprentissage

    Args:
        observed_mask (torch.tensor (B,K,L)): matrice masque des vraies observations
    
    Return :
        torch.tensor(B,K,L) : matrice conditionnelle m^co
    """
    # On génère nombres aléatoires sur la grille de observed_mask. En multipliant cela donne un tensor avec 0 les points non observés et nombre aléatoire
    #pour les points observés
    rand_for_mask = torch.rand_like(observed_mask) * observed_mask
    rand_for_mask = rand_for_mask.reshape(-1) # converti en vecteur 1D
    sample_ratio = np.random.rand()  # tirage aléatoire entre 0 et 1 = missin rate
    num_observed = observed_mask.sum().item() #compte le nombre total de points observés
    num_masked = round(num_observed * sample_ratio) #compte le nombre de points à masquer artificiellement
    rand_for_mask[rand_for_mask.topk(num_masked).indices] = -1 #on remplace les points par -1, sont les cibles d'imputation m^ta
    
    #remplace les valeurs non observés (0) et masqués (-1) par False et les valeurs observés(>0) par True. Redimensionne (B,K,L) et converti en float
    cond_mask = (rand_for_mask > 0).reshape(observed_mask.shape).float() 
    
    return cond_mask

# def get_hist_mask() a voir plus tard, autre méthode pour masquer les données



In [ ]:
def assemble_input_for_training(X_ori, indicating_mask, cond_mask, observed_tp):
    """
    Fonction de préparation des données pour l'entraînement. 
    Extrait les less tenseurs d'un batch, adapte leurs dimensions au réseau CSDI et construit le dictionnaire d'entrées

    Args:
        X_ori (): matrice des valeurs temporelles d'origine
        indicating_mask (): masque binaire des cibles d'imputation (m^ta)
        cond_mask (): masque binaire des observations conditionnelles (m^co)
        observed_tp (): les points des temps observés
    
    Return:
        Dict 
    """
    # permute (B,L,K) à (B,K,L) pour le réseau de débruitage
    inputs = {"X_ori" : X_ori.permute(0,2,1), 
              "indicated_mask" : indicating_mask.permute(0,2,1),
              "cond_mask" : cond_mask.permute(0,2,1),
              "observed_tp" : observed_tp}
    return inputs
    
def assemble_input_for_testing(X, cond_mask, observed_tp):
    """
    Fonction de préparation des données pour la test. Permet d'adapter aux dimension du réseau CSDI et construit un dictionnaire

    Args:
        X (_type_): _description_
        cond_mask (_type_): _description_
        observed_tp (_type_): _description_
    """
    inputs = {"X" : X.permute(0,2,1),
              "cond_mask" : cond_mask.permute(0,2,1),
              "observed_tp" : observed_tp}
    return input

def fit(train_set, val_set, file_type,n_sampling_times):
    """_summary_

    Args:
        train_set (Union[dict, str]): _description_
        val_set (Optional[Union[dict, str]]): _description_
        file_type (str): _description_
        n_sampling_times (int): _description_
    """
    # Etape 1 : préparation et chargement des données
    
    
    # Etape 2 : Boucle d'entraînement et sélection du meilleur modèle
    
    # Etape 3 : Sauvergarde du modèle 

In [ ]:
def CSDI():
    """
    Args:
        n_steps: int, number of time steps in the time-series data sample
        n_feature: int, number of the feature in the time-series data sample
        n_layers: int, number of layers in the CSDI model
        n_heads: int, number of heads in the multi-head attention mechanism
        n_channel: int, number of residual channels
        d_time_embedding: int, dimension number of the time embedding
        d_feature_embedding : int, dimension number of the feature embedding
        d_diffusion_embedding: int, dimension number of the diffusion embedding
        is_unconditional: bool,  weather the model is unconditional or conditional
        target_strategy: str, "mix" or "random", strategy to select the target for the diffusion process
        n_diffusion_steps: int, number of the diffusion step T in the original paper
        schedule: str, "quad" or "linear", schedule for other noise levels.
        beta_start: float, minimum noise level
        beta_end: float, maximum noise level
        batch_size: int, batch size for training and evaluating the model
        epochs: int, number of epochs for training the model 
        patience:[optional], int The patience for the early-stopping mechanism. Given a positive integer, the training process will be stopped when the model does not perform better after that number of epochs. Leaving it default as None will disable the early-stopping.
        optimizer: [optimizer, type = int] Adam by default, optimizer for model training
        num_workers: int, number of subprocesses to use for data loading. 0 if data loading will be in the main process
        device: "cpu" for now, gpu later maybe
        saving_path: str, path to save results
        model_saving_strategy:[optional] str, None, "best", "better", "all"
        verbose: bool, whether to print out the training logs during the training process
        
    
    """
